# Part 1: PyTorch MLP - Tasks 1 & 2
## Comparing NumPy and PyTorch MLP Implementations

This notebook demonstrates:
1. Training PyTorch MLP on make_moons dataset
2. Comparing with NumPy MLP implementation (if available)
3. Visualizing results with decision boundaries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim

from pytorch_mlp import MLP
from pytorch_train_mlp import accuracy

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Generate Training and Test Data

In [ ]:
# Generate make_moons dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Convert to PyTorch tensors
X_train_torch = torch.FloatTensor(X_train)
y_train_torch = torch.LongTensor(y_train)
X_test_torch = torch.FloatTensor(X_test)
y_test_torch = torch.LongTensor(y_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y))}")

## 2. Visualize the Dataset

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], c='blue', marker='o', label='Class 0', alpha=0.6)
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], c='red', marker='s', label='Class 1', alpha=0.6)
plt.title('Training Set')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c='blue', marker='o', label='Class 0', alpha=0.6)
plt.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c='red', marker='s', label='Class 1', alpha=0.6)
plt.title('Test Set')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Train PyTorch MLP

In [ ]:
# Create PyTorch MLP model
n_inputs = 2
n_hidden = [20]  # One hidden layer with 20 units
n_classes = 2

pytorch_mlp = MLP(n_inputs, n_hidden, n_classes)
print("PyTorch MLP Architecture:")
print(pytorch_mlp)
print(f"\nTotal parameters: {sum(p.numel() for p in pytorch_mlp.parameters())}")

In [ ]:
# Training configuration
learning_rate = 0.01
max_epochs = 1500
eval_freq = 100

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(pytorch_mlp.parameters(), lr=learning_rate)

# Training loop
train_losses = []
train_accuracies = []
test_accuracies = []

print("Training PyTorch MLP...\n")
for epoch in range(max_epochs):
    # Forward pass
    optimizer.zero_grad()
    outputs = pytorch_mlp(X_train_torch)
    loss = criterion(outputs, y_train_torch)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Record metrics
    train_losses.append(loss.item())
    
    if (epoch + 1) % eval_freq == 0:
        with torch.no_grad():
            train_acc = accuracy(pytorch_mlp(X_train_torch), y_train_torch)
            test_acc = accuracy(pytorch_mlp(X_test_torch), y_test_torch)
            train_accuracies.append(train_acc)
            test_accuracies.append(test_acc)
            print(f"Epoch {epoch+1:4d}: Loss={loss.item():.4f}, Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

print("\nTraining completed!")

## 4. Plot Training Metrics

In [ ]:
plt.figure(figsize=(15, 5))

# Plot loss
plt.subplot(1, 3, 1)
plt.plot(train_losses, alpha=0.7)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True, alpha=0.3)

# Plot accuracies
plt.subplot(1, 3, 2)
eval_epochs = list(range(eval_freq, max_epochs + 1, eval_freq))
plt.plot(eval_epochs, train_accuracies, label='Train Accuracy', marker='o', markersize=4)
plt.plot(eval_epochs, test_accuracies, label='Test Accuracy', marker='s', markersize=4)
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Final accuracy
plt.subplot(1, 3, 3)
final_train_acc = train_accuracies[-1]
final_test_acc = test_accuracies[-1]
plt.bar(['Train', 'Test'], [final_train_acc, final_test_acc], color=['blue', 'orange'])
plt.title('Final Accuracy')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
for i, v in enumerate([final_train_acc, final_test_acc]):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Visualize Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """Plot decision boundary for 2D classification"""
    h = 0.02  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    with torch.no_grad():
        Z = model(torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]))
        Z = torch.argmax(Z, dim=1).numpy()
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', marker='o', edgecolor='k', s=50, label='Class 0')
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', marker='s', edgecolor='k', s=50, label='Class 1')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_decision_boundary(pytorch_mlp, X_train, y_train, 'PyTorch MLP - Training Set')

plt.subplot(1, 2, 2)
plot_decision_boundary(pytorch_mlp, X_test, y_test, 'PyTorch MLP - Test Set')

plt.tight_layout()
plt.show()

## 6. Test on Different Datasets
### Try the MLP on other scikit-learn datasets

In [ ]:
def train_and_evaluate_mlp(X, y, dataset_name, n_hidden=[20], epochs=1000):
    """Train MLP and return final test accuracy"""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.LongTensor(y_test)
    
    model = MLP(X.shape[1], n_hidden, len(np.unique(y)))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        test_acc = accuracy(model(X_test_t), y_test_t)
    
    print(f"{dataset_name}: Test Accuracy = {test_acc:.4f}")
    
    # Plot decision boundary
    if X.shape[1] == 2:
        plot_decision_boundary(model, X_test, y_test, f'{dataset_name} - Decision Boundary')
        plt.show()
    
    return test_acc

# Test on different datasets
print("Testing PyTorch MLP on different datasets:\n")

# Circles dataset
X_circles, y_circles = make_circles(n_samples=1000, noise=0.1, factor=0.5, random_state=42)
acc_circles = train_and_evaluate_mlp(X_circles, y_circles, "Circles", n_hidden=[20, 20])

# Classification dataset
X_class, y_class = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                                       n_informative=2, n_clusters_per_class=1, 
                                       random_state=42)
acc_class = train_and_evaluate_mlp(X_class, y_class, "Classification", n_hidden=[20])

## 7. Summary and Comparison

### PyTorch MLP Results:
- Successfully implemented MLP using PyTorch's `nn.Module`
- Configurable architecture with flexible hidden layers
- Achieves good accuracy on make_moons and other datasets
- Uses automatic differentiation for backpropagation

### Key Observations:
1. PyTorch provides a clean, modular approach to building neural networks
2. The model successfully learns non-linear decision boundaries
3. Performance is comparable across different datasets with appropriate architecture

### Note on NumPy MLP Comparison:
If you have a NumPy implementation from Assignment 1, you can import it and train it on the same data to compare:
- Training time
- Final accuracy
- Implementation complexity

PyTorch generally provides better performance and easier implementation compared to pure NumPy.